## **Protocol 003 — Stage 3**: HADDOCK Score Extraction and iRMSD Computation

**Input:**
- HADDOCK output folder — one subfolder per signal peptide, each containing
  PDB files for all docking models across all clusters (typically 10 clusters × 4 models).
- Reference PDB files — 10 models of the experimental SecA/LamB-SP complex
  (PDB: 2VDA), used as the iRMSD reference.

**Output:**
- `best_matrix` — list of best-scoring clusters per SP, with mean/SD of HADDOCK
  score and energy components, plus minimum iRMSD vs. PDB:2VDA.
  Corresponds to Supplementary Table 3.

### Install Requirements and Import Libraries

In [ ]:
import numpy as np
import os
import pandas as pd

# pdb2sql is used to compute interface RMSD (iRMSD) between docking models
!pip install pdb2sql
from pdb2sql import StructureSimilarity

### Helper Functions

Two functions process the HADDOCK output:
- `initial_matrix`: reads a single PDB file, extracts energy terms from REMARK lines,
  and computes the HADDOCK score.
- `select_best_temp_row`: given all clusters for one SP, returns the cluster with
  the lowest mean HADDOCK score.

**HADDOCK score formula:**
`HHscore = 1.0×Evdw + 0.2×Eelec + 1.0×Edesol + 0.1×Eair`

In [ ]:
def Convert(string):
    """Split a string identifier by underscores into a list."""
    return list(string.split('_'))


def initial_matrix(id_uniprot, direction_pdb, name_subfichero):
    """
    Extract energy terms from a HADDOCK PDB REMARK block and compute the HADDOCK score.

    Args:
        id_uniprot (str): UniProt identifier of the signal peptide.
        direction_pdb (str): Full path to the PDB file.
        name_subfichero (str): Filename of the PDB (used to label the output row).

    Returns:
        list: [cluster, model_number, UniProt_ID, Evdw, Eelec, Eair,
               Edesol, Eburied, HHscore]
    """
    with open(direction_pdb, 'r') as f:
        listas_pdb = f.readlines()

    # Energy terms are stored in specific REMARK lines by HADDOCK
    row_en    = list(listas_pdb[6].split(', '))
    row_desol = list(listas_pdb[26].split(': '))
    row_bur   = list(listas_pdb[31].split(': '))

    Evdw    = float(row_en[5])         # Van der Waals energy
    Eelec   = float(row_en[6])         # Electrostatic energy
    Eair    = float(row_en[7])         # Ambiguous restraint energy
    Edesol  = float(row_desol[1][:-1]) # Desolvation energy
    Eburied = float(row_bur[1][:-1])   # Buried surface area energy

    # HADDOCK score: 1.0*Evdw + 0.2*Eelec + 1.0*Edesol + 0.1*Eair
    HHscore = round(Evdw + 0.2*Eelec + Edesol + 0.1*Eair, 3)

    core = Convert(name_subfichero[:-4])  # Remove .pdb extension
    core.extend([id_uniprot, Evdw, Eelec, Eair, Edesol, Eburied, HHscore])
    return core


def select_best_temp_row(temp_matrix):
    """
    Select the cluster with the lowest mean HADDOCK score from a set of clusters.

    Args:
        temp_matrix (list): List of rows, each representing one docking model.
                            Column 0 = cluster label, column 8 = HADDOCK score.

    Returns:
        list: Best cluster row containing cluster label, mean and SD of each
              energy term, and minimum iRMSD.
    """
    matrix = pd.DataFrame(temp_matrix)
    matrix[0] = matrix[0].astype('category')
    matrix[8] = matrix[8].astype('float')
    categ = list(matrix[0].cat.categories)

    best_vector = [0] * 17
    ref = [100] * 17

    for cluster in categ:
        rows = matrix.loc[matrix[0] == cluster]
        list_mean = [round(v, 3) for v in list(rows.mean())]
        list_std  = [round(v, 3) for v in list(rows.std())]
        vector = [cluster] + list_mean + list_std

        # Keep the cluster with the lowest mean HADDOCK score
        if vector[8] < ref[8]:
            best_vector = vector
        ref = best_vector

    return best_vector

### Output Column Reference

Each row in `best_matrix` contains the following fields:

| Column | Description |
|--------|-------------|
| `Name` | SP identifier (cluster + UniProt ID) |
| `ncluster` | Cluster number |
| `Evdw` / `sd_Evdw` | Van der Waals energy (mean ± SD) |
| `Eelec` / `sd_Eelec` | Electrostatic energy (mean ± SD) |
| `Eair` / `sd_Eair` | Ambiguous restraint energy (mean ± SD) |
| `Edesol` / `sd_Edesol` | Desolvation energy (mean ± SD) |
| `Eburied` / `sd_Eburied` | Buried surface area energy (mean ± SD) |
| `HHscore` / `sd_HHscore` | HADDOCK score (mean ± SD) |
| `irmsd` | Minimum iRMSD vs. PDB:2VDA across reference models |

### Process All HADDOCK Results

For each SP, iterate over all docking models, compute iRMSD against
the 10 reference models of PDB:2VDA, extract energy terms, and select
the best cluster per SP.

In [ ]:
best_matrix = []

# Reference PDB files: 10 models of the experimental SecA/LamB-SP complex (PDB:2VDA)
# Update this path to your local reference PDB folder
ref_pdb_dir = 'data/4_5_Input_clusters_PyDockEneRes_MAPIYA/secA_models_split'  # Update to your local path
ref = [f'seca_{str(i).zfill(3)}_secA_models_split.pdb' for i in range(1, 11)]

# Update this path to your local HADDOCK results folder
haddock_dir = 'data/HADDOCK_results'  # Update to your local path

with os.scandir(haddock_dir) as sp_folders:
    for sp_folder in sp_folders:
        path_sp = os.path.join(haddock_dir, sp_folder.name)
        temp_matrix = []

        with os.scandir(path_sp) as model_files:
            for model_file in model_files:
                path_model = os.path.join(path_sp, model_file.name)

                # Compute iRMSD against each of the 10 reference models
                list_irmsd = []
                for ref_file in ref:
                    sim = StructureSimilarity(path_model, os.path.join(ref_pdb_dir, ref_file))
                    list_irmsd.append(sim.compute_irmsd_pdb2sql())

                # Extract energy terms and append minimum iRMSD
                row = initial_matrix(sp_folder.name, path_model, model_file.name) + [min(list_irmsd)]
                temp_matrix.append(row)

        # Select the best cluster for this SP
        best_temp_row = [sp_folder.name] + select_best_temp_row(temp_matrix)
        best_matrix.append(best_temp_row)
        print(best_temp_row)

# Save results to CSV (Supplementary Table 3)
np.savetxt('SupplementaryTable3_IRMSD_HADDOCKscore.csv', best_matrix, delimiter=', ', fmt='%s')
print('Saved: SupplementaryTable3_IRMSD_HADDOCKscore.csv')

['C123_P0AEU0', 'cluster12', 308.5, -46.203, -15.686, 13.695, -26.114, 1358.722, -74.084, 0.446, 7.782, 5.912, 14.698, 2.127, 78.504, 7.11, 0.094]
['C124_P0AFH8', 'cluster14', 308.5, -45.916, -222.434, 73.366, -5.953, 1963.1, -89.019, 2.386, 18.808, 23.462, 22.671, 2.945, 351.774, 22.005, 0.377]
['C125_P0C8Z8', 'cluster6', 308.5, -45.481, -37.153, 16.002, -10.972, 1492.888, -62.283, 0.267, 5.397, 12.064, 16.831, 5.936, 144.124, 9.915, 0.052]
['C126_P24093', 'cluster3', 308.5, -48.319, -80.6, 22.738, -22.211, 1557.372, -84.376, 0.443, 5.176, 28.199, 20.035, 3.569, 163.37, 6.847, 0.042]
['C127_P28307', 'cluster1', 308.5, -46.479, -80.451, 33.215, -16.122, 1674.733, -75.37, 0.445, 4.325, 60.156, 13.518, 4.763, 92.521, 2.636, 0.219]
['C128_P28585', 'cluster3', 308.5, -48.591, -191.528, 22.898, -19.236, 2005.615, -103.843, 1.871, 9.948, 68.686, 22.471, 4.801, 238.224, 17.569, 0.223]
['C129_P30130', 'cluster11', 308.5, -37.944, -33.733, 42.203, -17.917, 1423.108, -58.387, 1.518, 4.514, 25.18